# X Fake Account Detection

## 1. Problem definition
This notebook develops a machine-learning prototype for an X/Twitter account risk-detection system.

The goal is not to definitively classify a person as fake, but to estimate whether an account shows suspicious or anomalous behavioral patterns based on profile, activity, engagement, and content features.

Outputs include a risk probability and risk level.

## 2. Dataset loading
Load the dataset (initially synthetic for development).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# Load dataset
data_path = '../dataset/accounts.csv'
df = pd.read_csv(data_path)
df.head()

## 3. Data exploration
Understand feature distributions and class balance.

In [ ]:
print(df.info())
print('\nClass balance:')
print(df['label'].value_counts(normalize=True))

In [ ]:
sns.countplot(x='label', data=df)
plt.title('Distribution of Legit (0) vs Suspicious (1)')
plt.show()

## 4. Data cleaning
Handle any missing values. (Synthetic data is clean, but in production, we fill or drop appropriately).

In [ ]:
# Example: Fill missing values if any
# df.fillna(0, inplace=True)
print('Missing values:\n', df.isnull().sum().sum())

## 5. Feature engineering
The dataset already contains the engineered features (e.g., `followers_following_ratio`). Here we separate features and the target variable.

In [ ]:
features = [
    'account_age_days', 'followers_count', 'following_count', 'post_count', 
    'followers_following_ratio', 'profile_completeness', 'verified',
    'posts_per_day', 'average_post_interval', 'posting_interval_std', 
    'reply_ratio', 'repost_ratio', 'original_post_ratio', 'activity_burst_score',
    'average_likes', 'average_replies', 'average_reposts', 'average_quotes', 
    'engagement_rate', 'engagement_variance',
    'duplicate_content_ratio', 'average_text_length', 'hashtag_frequency', 
    'mention_frequency', 'url_frequency', 'repeated_hashtag_ratio'
]

X = df[features]
y = df['label']

## 6. Feature correlation/analysis
Visualize correlations between features and the target label.

In [ ]:
plt.figure(figsize=(15, 12))
corr = df[features + ['label']].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

## 7. Train/test split
Split the data into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 8. Baseline models
Train Logistic Regression and Random Forest as baselines.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Scale data for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)
print('Logistic Regression F1:', f1_score(y_test, lr_preds))

# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print('Random Forest F1:', f1_score(y_test, rf_preds))

## 9. XGBoost/LightGBM model
Compare with gradient boosting (using XGBoost here).

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
print('XGBoost F1:', f1_score(y_test, xgb_preds))

## 10. Model evaluation
Detailed evaluation of the best model (XGBoost). Focus on Precision/Recall and False Positives.

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

print('--- XGBoost Evaluation ---')
print('Accuracy:', accuracy_score(y_test, xgb_preds))
print('Precision:', precision_score(y_test, xgb_preds))
print('Recall:', recall_score(y_test, xgb_preds))
print('F1-score:', f1_score(y_test, xgb_preds))
print('ROC-AUC:', roc_auc_score(y_test, xgb_probs))

precision, recall, _ = precision_recall_curve(y_test, xgb_probs)
pr_auc = auc(recall, precision)
print('PR-AUC:', pr_auc)

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, xgb_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 11. Feature importance
Analyze which signals contribute most to the risk score.

In [ ]:
importance = pd.DataFrame({'feature': features, 'importance': xgb_model.feature_importances_})
importance = importance.sort_values('importance', ascending=False)
print(importance.head(10))

plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=importance.head(15))
plt.title('Top 15 Feature Importances (XGBoost)')
plt.show()

## 12. Error analysis
Examine false positives to understand why legitimate accounts might be flagged.

In [ ]:
X_test_eval = X_test.copy()
X_test_eval['true_label'] = y_test
X_test_eval['pred_label'] = xgb_preds
X_test_eval['risk_probability'] = xgb_probs

false_positives = X_test_eval[(X_test_eval['true_label'] == 0) & (X_test_eval['pred_label'] == 1)]
print(f'Found {len(false_positives)} false positives.')
if len(false_positives) > 0:
    display(false_positives.head())

## 13. Save the best model
Export the model and the feature schema for FastAPI production use.

In [ ]:
import joblib

os.makedirs('../models', exist_ok=True)
model_path = '../models/x_account_risk_model.pkl'
joblib.dump(xgb_model, model_path)

schema = {
    'version': '1.0',
    'features': features
}
with open('../models/feature_schema.json', 'w') as f:
    json.dump(schema, f, indent=4)
    
print('Model and schema saved to ml/models/')